In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

## $\text{\textcolor{green}{Day-long Horizon Solver w/ Curtailment}}$


In [ ]:
# Daily solver
def solve_battery_day(day_df, minimum_self_sufficiency, charge_eff, discharge_eff, dT, C_rate, load_profile):
    day_df = day_df.sort_index()

    PV_gen = day_df["PV_total"].to_numpy()
    Wind_gen = day_df["Wind_total"].to_numpy()
    n = len(day_df)

    Pload_np = load_profile

    # Big-M from generation only
    P_capacity_max = 1.05 * np.max(PV_gen + Wind_gen)

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)

    # Variables
    P_capacity = cp.Variable(nonneg=True)
    E_capacity = cp.Variable(nonneg=True)

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)
    z = cp.Variable(n, boolean=True)

    Pgrid_buy = cp.Variable(n, nonneg=True)
    SOC = cp.Variable(n)
    Pcurtail = cp.Variable(n, nonneg= True)

    constraints = [
        P_capacity <= P_capacity_max,
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        Pcharge <= P_capacity_max * z,
        Pdischarge <= P_capacity_max * (1 - z),

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Curtailment can never exceed available generation
        Pcurtail <= Pgen_PV + Pgen_W,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge + Pcurtail,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity,

        cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT
    ]

    # Extras for curtailment -- penalty to avoid spilling and tracker of spillage
    spilled_energy = cp.sum(Pcurtail) * dT
    eps_curt = 1e-6

    problem = cp.Problem(cp.Minimize(E_capacity + eps_curt * spilled_energy), constraints)

    try:
        problem.solve(solver=cp.CBC, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            return {
            "status": problem.status,
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "spilled_energy_kWh": np.nan,
            "grid_and_curtail_simultaneous": np.nan
            }

        both_active = np.any(
            (Pgrid_buy.value > 1e-6) & (Pcurtail.value > 1e-6)
        )

        return {
            "status": problem.status,
            "E_capacity_kWh": E_capacity.value,
            "P_capacity_kW": P_capacity.value,
            "spilled_energy_kWh": np.sum(Pcurtail.value) * dT,
            "grid_and_curtail_simultaneous": both_active
        }

    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}",
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "spilled_energy_kWh": np.nan,
            "grid_and_curtail_simultaneous": np.nan
        }

## $\text{\textcolor{green}{Day-long Horizon for Battery Sizing w/ Variable Load \& Curtailment}}$


In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"
df = pd.read_csv(DATA_DIR,delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")
df_load = pd.read_csv(ROOT / "data/archive/archive/building_consumption.csv", parse_dates=["timestamp"])

filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14") &
    (df_load["timestamp"] <= "2022-04-11")
]
# How many 15-min readings are there available per meter_id?
counts = filtered.groupby("meter_id").size().reset_index(name="num_readings")

In [ ]:
total_load = filtered.groupby("timestamp")["consumption"].sum().reset_index(name="total_consumption")
total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean()

In [ ]:
# Confirm data has 48-step days
total_load_30min["date"] = total_load_30min.index.date
load_counts = total_load_30min.groupby("date").size()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[total_load_30min["date"].isin(load_complete_days)].copy()

# Make days into 48-step vectors

load_day_profiles = []

for _, day_df in load_days_df.groupby("date"):
    day_profile = day_df.sort_index()["total_consumption"].to_numpy()
    load_day_profiles.append(day_profile)

# print(load_day_profiles)

# Visualization of daily profiles <-- Would the MW scale be per time step or across the day profile?

daily_totals = total_load_30min.groupby(total_load_30min.index.date)["total_consumption"].sum()
print(daily_totals.head())
daily_energy = daily_totals * 0.5
print(daily_energy.describe())

In [ ]:
# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 0.5

$\text{\textcolor{purple}{I'm only getting 6 results for an optimal battery size across all self-sufficiencies, i.e. a single pair of generation and load days is able to meet any and all of the self-sufficiency goals}}$

I have 330 full-length generation days and 332 full-length load days

In [ ]:
# How does my generation behave compared to my load?
        # Daily Renewable Generation vs Daily Load

gen_daily = df_agg.groupby("date").apply(
    lambda x: (x["PV_total"].sum() + x["Wind_total"].sum()) * dT
)

load_daily = pd.Series([
    np.sum(profile) * dT for profile in load_day_profiles
])

print("Generation daily energy [kWh]")
print(gen_daily.describe())

print("\nLoad daily energy [kWh]")
print(load_daily.describe())


        # Theoretical maximum self-sufficiency

gen_days = list(df_agg.groupby("date"))

max_possible = []

for i, (day, day_df) in enumerate(gen_days):
    load_profile = load_day_profiles[i % len(load_day_profiles)]

    renewable_energy = (
        day_df["PV_total"].sum() + day_df["Wind_total"].sum()
    ) * dT

    load_energy = np.sum(load_profile) * dT

    max_ss = renewable_energy / load_energy

    max_possible.append({
        "date": day,
        "renewable_kWh": renewable_energy,
        "load_kWh": load_energy,
        "max_possible_self_sufficiency": max_ss
    })

max_possible_df = pd.DataFrame(max_possible)

print("Maximum Possible SElf-Sufficiency")
print(max_possible_df["max_possible_self_sufficiency"].describe())

In [ ]:
targets = np.linspace(0.5, 1.0, 6)
results = []

gen_days = list(df_agg.groupby("date"))

for target in targets:
    for i, (day, day_df) in enumerate(gen_days):
        if len(day_df) != 48:
            continue

        load_profile = load_day_profiles[i % len(load_day_profiles)]

        out = solve_battery_day(day_df, minimum_self_sufficiency=target,charge_eff=0.95, discharge_eff=0.95, dT=0.5, C_rate=0.5,load_profile=load_profile)
        out["date"] = pd.to_datetime(day)
        out["self_sufficiency"] = target
        results.append(out)

results_df = pd.DataFrame(results)

print(results_df)
print(results_df["status"].value_counts(dropna=False))
print(results_df["grid_and_curtail_simultaneous"].value_counts())

In [ ]:
valid = results_df[results_df["status"] == "optimal"].copy()

print("Total runs:", len(results_df))
print("Optimal runs:", len(valid))

summary = valid.groupby("self_sufficiency").agg(
    median_E=("E_capacity_kWh", "median"),
    p90_E=("E_capacity_kWh", lambda x: x.quantile(0.9)),
    max_E=("E_capacity_kWh", "max"),
    median_P=("P_capacity_kW", "median"),
    p90_P=("P_capacity_kW", lambda x: x.quantile(0.9)),
    max_P=("P_capacity_kW", "max"),
    median_spilled=("spilled_energy_kWh", "median"),
    p90_spilled=("spilled_energy_kWh", lambda x: x.quantile(0.9)),
    max_spilled=("spilled_energy_kWh", "max"),
    feasible_days=("E_capacity_kWh", "count")
).reset_index()

print(summary)

### $\text{\textcolor{green}{Plot for Best Battery Size vs. Self-Sufficiency}}$

$\text{\textcolor{green}{Sizing According to Energy Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_E"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_E"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_E"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required energy capacity [kWh]")
plt.title("Battery energy size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Sizing According to Power Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_P"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_P"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_P"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required power capacity [kW]")
plt.title("Battery power size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Distribution of Battery Sizes}}$

$\text{\textcolor{green}{Energy Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = valid_plot["self_sufficiency"] * 100
valid_plot.boxplot(column="E_capacity_kWh", by="self_sufficiency_pct")

plt.title("Distribution of daily energy capacity by self-sufficiency target")
plt.suptitle("")  # removes automatic pandas title

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Energy capacity [kWh]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Power Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = valid_plot["self_sufficiency"] * 100
valid_plot.boxplot(column="P_capacity_kW", by="self_sufficiency_pct")

plt.title("Distribution of daily power capacity by self-sufficiency target")
plt.suptitle("")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Power capacity [kW]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Spillage According to Self-Sufficiency}}$

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(summary["self_sufficiency"] * 100, summary["median_spilled"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_spilled"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_spilled"],marker="o", label="Maximum")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Spilled renewable energy [kWh/day]")
plt.title("Renewable curtailment vs self-sufficiency target")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Why can spillage still happen if the battery could, in theory, absorb that energy?

## $\text{\textcolor{green}{Day-long Horizon for Battery Sizing w/ Fixed Load \& Curtailment}}$

Exploration Studies:
- input power as a daily profile => you want to get a distribution of battery capacities
    - would this distribution also have to vary according to self-sufficiencies?
- limit P capacity in reference to C capacity => with C-rate or a weight called something arbitrary
    - different c-rates => compare battery technologies/types of lithium ion batteries in energy systems

Small size of battery could be due to:
- enough generation to not require battery
- lax constraint on self-sufficiency

Make sure that when you're charging your'e not discharding
    - maybe introduce boolean variable to prevent simultaneous operation

target function could be to minimize SOC * battery E capacity * 1.0 or just the E_capacity
you need to make the 

Meeting April 21st Notes
- Change data into daily profiles => you want to get a distribution of battery sizes depending on the profile of days
    *modify daily profile function from daily profile clustering + incorporate aggregation function.
- Start optimization with real data to have comparison in terms of accuracy
    - assume you have next-day's weather and power values
- Upon having comparison, you can incorporate clustering results
- optimization code should be wrapped in a function
    - input: 24-hour of real pv power and real wind
    - output: schedule for battery operation

- Load data may exist in Gaggle or we can simulate it from DTU's load data
    - doesn't overlap directly timewise => don't take average so you have a spikier profile

In [ ]:
# Daily solver
def solve_battery_day(day_df, minimum_self_sufficiency, charge_eff, discharge_eff, dT, C_rate, load_profile):
    day_df = day_df.sort_index()

    PV_gen = day_df["PV_total"].to_numpy()
    Wind_gen = day_df["Wind_total"].to_numpy()
    n = len(day_df)

    Pload_np = load_profile

    # Big-M from generation only
    P_capacity_max = 1.05 * np.max(PV_gen + Wind_gen)

    # Constants
    Pload = cp.Constant(Pload_np)
    Pgen_PV = cp.Constant(PV_gen)
    Pgen_W = cp.Constant(Wind_gen)

    # Variables
    P_capacity = cp.Variable(nonneg=True)
    E_capacity = cp.Variable(nonneg=True)

    Pcharge = cp.Variable(n, nonneg=True)
    Pdischarge = cp.Variable(n, nonneg=True)
    z = cp.Variable(n, boolean=True)

    Pgrid_buy = cp.Variable(n, nonneg=True)
    SOC = cp.Variable(n)
    Pcurtail = cp.Variable(n, nonneg= True)

    constraints = [
        P_capacity <= P_capacity_max,
        P_capacity <= C_rate * E_capacity,

        Pcharge <= P_capacity,
        Pdischarge <= P_capacity,

        Pcharge <= P_capacity_max * z,
        Pdischarge <= P_capacity_max * (1 - z),

        SOC >= 0.1 * E_capacity,
        SOC <= E_capacity,

        # Curtailment can never exceed available generation
        Pcurtail <= Pgen_PV + Pgen_W,

        # Power balance 
        Pgen_PV + Pgen_W + Pdischarge + Pgrid_buy == Pload + Pcharge + Pcurtail,

        SOC[0] == 0.5 * E_capacity + (Pcharge[0] * charge_eff - Pdischarge[0] / discharge_eff) * dT,
        SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff) * dT,

        SOC[n - 1] >= 0.5 * E_capacity,

        cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT
    ]

    # Extras for curtailment -- penalty to avoid spilling and tracker of spillage
    spilled_energy = cp.sum(Pcurtail) * dT
    eps_curt = 1e-6

    problem = cp.Problem(cp.Minimize(E_capacity + eps_curt * spilled_energy), constraints)

    try:
        problem.solve(solver = cp.CBC, verbose=False) # or cp.HIGHS <-- it's also installed!
        return {
            "status": problem.status,
            "E_capacity_kWh": E_capacity.value,
            "P_capacity_kW": P_capacity.value,
            "spilled_energy_kWh": spilled_energy.value,
        }
    except Exception as e:
        return {
            "status": f"solver_error: {type(e).__name__}",
            "E_capacity_kWh": np.nan,
            "P_capacity_kW": np.nan,
            "spilled_energy_kWh": np.nan,
        }

In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"
df = pd.read_csv(DATA_DIR,delimiter=",",decimal=".",parse_dates=["ts"],index_col="ts")

# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg["Wind_total"] = df_agg[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

# Keep only complete days
df_agg["date"] = df_agg.index.date
counts_per_day = df_agg.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index
df_agg = df_agg[df_agg["date"].isin(complete_days)].copy()

# Settings
dT = 0.5
charge_eff = 0.95
discharge_eff = 0.95
C_rate = 0.5

In [ ]:
# Loop over self-sufficiency targets
targets = np.linspace(0.5, 1.0, 6)   # 50%, 60%, ..., 100%

results = []

for target in targets:
    for day, day_df in df_agg.groupby("date"):
        if len(day_df) != 48:
            continue

        out = solve_battery_day(day_df, minimum_self_sufficiency=target, charge_eff=0.95, discharge_eff=0.95, dT=0.5, C_rate=0.5,load_profile= (np.zeros(len(day_df)) + 2.0))
        out["date"] = pd.to_datetime(day)
        out["self_sufficiency"] = target
        results.append(out)

#day_df, minimum_self_sufficiency, charge_eff, discharge_eff, dT, C_rate load_profile

results_df = pd.DataFrame(results)

print(results_df.head())
print(results_df["status"].value_counts(dropna=False))

In [ ]:
valid = results_df[results_df["status"] == "optimal"].copy()

summary = valid.groupby("self_sufficiency").agg(
    median_E=("E_capacity_kWh", "median"),
    p90_E=("E_capacity_kWh", lambda x: x.quantile(0.9)),
    max_E=("E_capacity_kWh", "max"),
    median_P=("P_capacity_kW", "median"),
    p90_P=("P_capacity_kW", lambda x: x.quantile(0.9)),
    max_P=("P_capacity_kW", "max"),
    median_spilled=("spilled_energy_kWh", "median"),
    p90_spilled=("spilled_energy_kWh", lambda x: x.quantile(0.9)),
    max_spilled=("spilled_energy_kWh", "max"),
    feasible_days=("E_capacity_kWh", "count")
).reset_index()

print(summary)

### $\text{\textcolor{green}{Plot for Best Battery Size vs. Self-Sufficiency}}$

$\text{\textcolor{green}{Sizing According to Energy Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_E"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_E"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_E"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required energy capacity [kWh]")
plt.title("Battery energy size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Sizing According to Power Capacity}}$

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(summary["self_sufficiency"] * 100, summary["median_P"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_P"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_P"], marker="o", label="Maximum")
plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Required power capacity [kW]")
plt.title("Battery power size vs self-sufficiency target")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Distribution of Battery Sizes}}$

$\text{\textcolor{green}{Energy Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = valid_plot["self_sufficiency"] * 100
valid_plot.boxplot(column="E_capacity_kWh", by="self_sufficiency_pct")

plt.title("Distribution of daily energy capacity by self-sufficiency target")
plt.suptitle("")  # removes automatic pandas title

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Energy capacity [kWh]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

$\text{\textcolor{green}{Power Capacity Distribution}}$

In [ ]:
plt.figure(figsize=(9, 4))

valid_plot = valid.copy()
valid_plot["self_sufficiency_pct"] = valid_plot["self_sufficiency"] * 100
valid_plot.boxplot(column="P_capacity_kW", by="self_sufficiency_pct")

plt.title("Distribution of daily power capacity by self-sufficiency target")
plt.suptitle("")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Power capacity [kW]")

plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### $\text{\textcolor{green}{Plot for Spillage According to Self-Sufficiency}}$

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(summary["self_sufficiency"] * 100, summary["median_spilled"], marker="o", label="Median")
plt.plot(summary["self_sufficiency"] * 100, summary["p90_spilled"], marker="o", label="90th percentile")
plt.plot(summary["self_sufficiency"] * 100, summary["max_spilled"],marker="o", label="Maximum")

plt.xlabel("Minimum self-sufficiency [%]")
plt.ylabel("Spilled renewable energy [kWh/day]")
plt.title("Renewable curtailment vs self-sufficiency target")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## $\text{\textcolor{green}{Year-long Horizon for Battery Sizing}}$

In [ ]:
cp.MAX_NUM_SUBEXPRESSIONS = 10000

- Change data into daily profiles => you want to get a distribution of battery sizes depending on the profile of days
    *modify daily profile function from daily profile clustering + incorporate aggregation function.

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"
df = pd.read_csv(DATA_DIR, 
                     delimiter=',', 
                     decimal='.', 
                     parse_dates=['ts'],  
                     index_col='ts')

# Aggregate df across PV modules and wind turbines
df_agg = df.copy()
df_agg["PV_total"] = df_agg[["B117_m", "B319_m", "B330_1_m","B330_2_m","B330_3_m", "B716_m", "B715_m"]].sum(axis=1, min_count=1)
df_agg["Wind_total"] = df_agg[["Aircon_WT Power_m", "Gaia_WT Power_m"]].sum(axis=1, min_count= 1)
df_agg.dropna(subset=["PV_total", "Wind_total"], how='any', inplace=True)

# Transform generation data into numeric arrays
PV_gen = df_agg['PV_total'].to_numpy()
Wind_gen = df_agg['Wind_total'].to_numpy()
# Check before building optimization
print("PV NaNs:", np.isnan(PV_gen).sum())
print("Wind NaNs:", np.isnan(Wind_gen).sum())

n = len(df_agg) 
print(n)

#current n size assumes we have info for a full year; n should span a single day with timesteps depending on the time resolution of the optimization (choice of smaller horizon should be explained);
#reduces complexity

In [ ]:
## Defining a constant 10 kW load
Pload_np = np.zeros((n))+ 2
Pload = cp.Constant(Pload_np)
dT = 0.5 #h ; because df is in 30 mins resolution

## Target min self-sufficiency
minimum_self_sufficiency = 0.8

## Battery parameters; power and energy capacities are different values :) -- efficiency refers to power, always
charge_eff = 0.95
discharge_eff = 0.95
P_capacity = cp.Variable(nonneg= True)
E_capacity = cp.Variable(nonneg=True)

P_capacity_max = 20 #kW --Chatty suggests setting up this upper bound to be able to use the boolean for on/off discharging; is it reasonable
C_rate = 0.5 #e.g. a 2 hour batter to avoid having an abnormally large power capacity

## Defining generation values from Syslab Data -- separate into solar and wind generation
Pgen_PV = cp.Constant(PV_gen)
Pgen_W = cp.Constant(Wind_gen)
#print(df_agg['PV_total'].describe(), df_agg['Wind_total'].describe())

## Defining charge and discharge power values
Pcharge = cp.Variable(n,nonneg=True) # charge is positive
Pdischarge = cp.Variable(n,nonneg=True) # discharge is positive
z = cp.Variable(n, boolean = True) #boolean to prevent charging and discharging from happening simultaneously.

## Defining power imported and exported to the grid -- becomes 0 with 100% self-sufficiency
Pgrid_buy = cp.Variable(n, nonneg=True)
#Pgrid_sell = cp.Variable(n, nonneg=True) We're not selling anumore

## Defining variable SOC
SOC = cp.Variable(n)

#Curtailment for energy that doesn't get stored
Pcurtail = cp.Variable(n, nonneg=True)

## Defining equation for self-sufficiency -- we might need to revise the self-sufficiency definition
#self_sufficiency = (Pgen_PV + Pgen_W - Pcharge)/Pload 
#average_self_sufficiency = sum(self_sufficiency)/n


In [ ]:
#To avoid simultaneous charge and discharge
constraints = [P_capacity <= P_capacity_max,
               P_capacity <= C_rate * E_capacity,
               Pcharge <= P_capacity,
               Pdischarge <= P_capacity,
               Pcharge <= P_capacity_max * z,
               Pdischarge <= P_capacity_max * (1 - z)]
constraints += [SOC >= E_capacity * 0.1, SOC <= E_capacity] # SOC ranges between 10 to 100% across a day
constraints += [Pgen_PV + Pgen_W + Pdischarge - Pcharge - Pcurtail + Pgrid_buy  == Pload, # power flow equation 
               SOC[0] == 0.5 * E_capacity + (Pcharge[0]*charge_eff - Pdischarge[0]/ discharge_eff)*dT, ## Starting with 50% charge
               SOC[1:] == SOC[:-1] + (Pcharge[1:] * charge_eff - Pdischarge[1:] / discharge_eff)*dT, # battery state definition
               SOC[n-1] >= 0.5 * E_capacity] # minimum 50% charge

#Constraint for self-sufficiency, which caps the amount of energy that can be imported at 20% of the load's demand => forces reliance on a battery
constraints += [cp.sum(Pgrid_buy) * dT <= (1 - minimum_self_sufficiency) * np.sum(Pload_np) * dT]
               
               

In [ ]:
problem2 = cp.Problem(cp.Minimize(E_capacity), constraints) #minimize
problem2.solve(solver = cp.CBC,verbose=True)

In [ ]:
total_load = np.sum(Pload_np) * dT
total_gen = np.sum(PV_gen + Wind_gen) * dT
print(total_load, total_gen)